[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke06-klinisk-praksis/02_klinisk_beslutningsstøtte_terskler_og_avveininger.ipynb)

# 🩺 Klinisk beslutningsstøtte: terskler og avveininger

I forrige notebook bygget vi en modell som anslo risiko. I denne notebooken tar vi neste steg: Hvordan bruker vi et slikt risikoanslag i praksis? En sannsynlighet i seg selv er ikke en beslutning. Først når vi bestemmer hva som skal skje ved lav, middels eller høy risiko, blir modellen relevant for klinisk arbeid.

Eksempelet under er også syntetisk, slik at vi kan utforske terskelvalg og konsekvenser uten å bruke ekte pasientdata.

## Læringsmål
- forklare forskjellen mellom en prediksjon og en beslutning
- forstå hvordan terskelvalg påvirker sensitivitet og spesifisitet
- diskutere kliniske konsekvenser av falske positive og falske negative
- reflektere over når en modell er nyttig som beslutningsstøtte i klinisk praksis

### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab

In [ ]:
import sys, subprocess, os
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokalt miljø")

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print("✅ Miljø klart")

## Prediksjon er ikke det samme som beslutning

En modell kan gi en **risiko** på for eksempel 0,28 for at en pasient tilhører en gruppe med økt sannsynlighet for sykdom eller behov for tettere oppfølging. Men modellen sier ikke automatisk hva klinikeren bør gjøre.

For å bruke modellen i praksis må vi velge en **terskel**:

- over terskelen: vi handler, for eksempel bestiller videre utredning, prøver eller raskere kontroll
- under terskelen: vi avventer, følger opp på annen måte eller lar klinisk vurdering veie tyngre

Terskelen er derfor et klinisk valg, ikke bare et teknisk valg. Den uttrykker hvor mye risiko vi er villige til å akseptere før systemet foreslår handling.

In [ ]:
# Syntetisk eksempel: samme risikomodell, ulike terskler

np.random.seed(42)
n = 500
sann_risiko = np.clip(np.random.beta(2, 5, size=n), 0, 1)
utfall = np.random.binomial(1, sann_risiko)
predikert_risiko = np.clip(sann_risiko + np.random.normal(0, 0.08, size=n), 0, 1)


def confusion_counts(y_true, y_prob, terskel):
    y_pred = (y_prob >= terskel).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, fp, tn, fn

terskler = [0.20, 0.35, 0.50]
for terskel in terskler:
    tp, fp, tn, fn = confusion_counts(utfall, predikert_risiko, terskel)
    sensitivitet = tp / max(tp + fn, 1)
    spesifisitet = tn / max(tn + fp, 1)
    print(f"Terskel {terskel:.2f}")
    print(f"  TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    print(f"  Sensitivitet={sensitivitet:.2f}, spesifisitet={spesifisitet:.2f}\n")

In [ ]:
# Visualisering av avveining mellom sensitivitet og spesifisitet

terskel_grid = np.linspace(0.05, 0.80, 40)
sens = []
spec = []
for terskel in terskel_grid:
    tp, fp, tn, fn = confusion_counts(utfall, predikert_risiko, terskel)
    sens.append(tp / max(tp + fn, 1))
    spec.append(tn / max(tn + fp, 1))

plt.plot(terskel_grid, sens, label='Sensitivitet')
plt.plot(terskel_grid, spec, label='Spesifisitet')
plt.xlabel('Terskel')
plt.ylabel('Andel')
plt.title('Valg av terskel påvirker klinisk avveining')
plt.legend()
plt.tight_layout()
plt.show()

## Klinisk tolkning

En lav terskel vil ofte gi:

- høyere sensitivitet
- flere falske positive
- flere pasienter til videre utredning eller kontroll

En høy terskel vil ofte gi:

- høyere spesifisitet
- flere falske negative
- færre unødige undersøkelser, men større risiko for å overse pasienter som burde vært fanget opp

I klinisk praksis finnes det sjelden én "riktig" terskel. Valget avhenger av hvilken tilstand det gjelder, hvor alvorlig det er å overse sykdom, hva slags tiltak en positiv anbefaling utløser, og hvor mye kapasitet tjenesten faktisk har.

Derfor bør terskelvalg ses som en del av implementeringen av modellen, ikke som en liten teknisk detalj etter at modellen er ferdig.

### Refleksjon
- Hvor lav kan terskelen være før systemet skaper for mange unødige tiltak eller henvisninger?
- I hvilke kliniske situasjoner er falske negative mer alvorlige enn falske positive, og hvorfor?
- Hvordan ville terskelvalget endre seg dersom tiltaket er billig og ufarlig versus kostbart eller belastende?
- Hvem bør egentlig fastsette terskelen: utvikleren, klinikeren, fagmiljøet eller institusjonen?
- Hvordan henger denne notebooken sammen med den forrige: Hva må være på plass både i modellen og i tjenesten før et risikoanslag kan brukes som beslutningsstøtte?